In [7]:
# Cell 1: Cấu hình mô phỏng ⚙️
import torch

class SimulationConfig:
    # === 1. Cấu hình kiến trúc 3-Tầng ===
    NUM_EDGE = 5               # Số lượng Edge Server (Tầng 2)
    GLOBAL_ROUNDS = 10         # Số vòng giao tiếp giữa Cloud và Edge (Cloud-Edge)
    EDGE_ROUNDS = 4            # Số vòng giao tiếp giữa Edge và Client (Edge-Client)
    
    # === 2. Cấu hình thuật toán và huấn luyện Client ===
    ALGORITHM = 'fedprox'      # 'fedavg', 'fedprox'
    LOCAL_EPOCHS = 5
    BATCH_SIZE = 32
    LEARNING_RATE = 0.01
    FEDPROX_MU = 0.01

    # === 3. Cấu hình hệ thống và đường dẫn ===
    CONFIG_PATH = '../configs/config.yaml'
    DATA_PATH = '../data'
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEED = 42

config = SimulationConfig()

In [8]:
# Cell 2: Tải cấu hình, dữ liệu và phân chia Client cho Edge 📦
import yaml, torch, numpy as np, time, copy, os
import matplotlib.pyplot as plt
from torch import nn
import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from src.dataset_preparation import prepare_data 
from src.monitoring_utils import ResourceMonitor, shutdown_pynvml

# Thiết lập seed
torch.manual_seed(config.SEED); np.random.seed(config.SEED)

# Đọc file config.yaml
print(f"Đang đọc file cấu hình từ: {config.CONFIG_PATH}")
with open(config.CONFIG_PATH, 'r') as f:
    loaded_config = yaml.safe_load(f)
data_params = loaded_config['data_config']
print(f"Đã tải cấu hình dữ liệu: {data_params}")

# Chuẩn bị dữ liệu (tải toàn bộ client về)
client_loaders, test_loader, class_names = prepare_data(
    data_path=config.DATA_PATH,
    batch_size=config.BATCH_SIZE,
    seed=config.SEED,
    **data_params
)
total_num_clients = len(client_loaders)
print(f"\nĐã tải dữ liệu cho tổng số {total_num_clients} client.")

# --- PHÂN CHIA CLIENT CHO CÁC EDGE SERVER ---
# Chia danh sách client_loaders thành các nhóm cho mỗi edge server
# np.array_split sẽ xử lý việc chia không đều một cách tự động
client_groups_for_edges = np.array_split(client_loaders, config.NUM_EDGE)
print(f"Đã chia {total_num_clients} client cho {config.NUM_EDGE} edge server:")
for i, group in enumerate(client_groups_for_edges):
    print(f"  - Edge Server {i}: có {len(group)} client")

print(f"Sử dụng thiết bị: {config.DEVICE}")

Đang đọc file cấu hình từ: ../configs/config.yaml
Đã tải cấu hình dữ liệu: {'dataset_name': 'cifar10', 'distribution_mode': 'non_iid_shards', 'num_clients': 10, 'shards_per_client': 2, 'dirichlet_alpha': 0.5}

Đã tải dữ liệu cho tổng số 10 client.
Đã chia 10 client cho 5 edge server:
  - Edge Server 0: có 2 client
  - Edge Server 1: có 2 client
  - Edge Server 2: có 2 client
  - Edge Server 3: có 2 client
  - Edge Server 4: có 2 client
Sử dụng thiết bị: cuda


In [9]:
# Cell 3: Định nghĩa mô hình Neural Network 🧠
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, input_size=32):
        super(SimpleCNN, self).__init__()
        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2), # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 16 -> 8
        )
        feature_size = (input_size // 4) ** 2 * 64
        self.fc_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feature_size, 512), nn.ReLU(),
            nn.Linear(512, num_classes)
        )
    def forward(self, x): return self.fc_stack(self.conv_stack(x))

In [10]:
# Cell 4: Logic cốt lõi của Federated Learning (Cập nhật) 🤖

# --- Hàm huấn luyện Client (Không thay đổi) ---
def client_update(client_loader, model):
    monitor = ResourceMonitor(); monitor.start()
    start_time = time.time(); local_model = copy.deepcopy(model).to(config.DEVICE); local_model.train();
    optimizer = torch.optim.SGD(local_model.parameters(), lr=config.LEARNING_RATE, momentum=0.9); criterion = nn.CrossEntropyLoss();
    for _ in range(config.LOCAL_EPOCHS):
        for data, target in client_loader:
            data, target = data.to(config.DEVICE), target.to(config.DEVICE)
            optimizer.zero_grad(); output = local_model(data); loss = criterion(output, target);
            if config.ALGORITHM == 'fedprox':
                proximal_term = sum((local_w - global_w).norm(2)**2 for local_w, global_w in zip(local_model.parameters(), model.parameters()))
                loss += (config.FEDPROX_MU / 2) * proximal_term
            loss.backward(); optimizer.step()
            monitor.track()
    comp_cost = time.time() - start_time
    return local_model.state_dict(), comp_cost, monitor.stop()

# --- Hàm tổng hợp (Chung cho cả Edge và Cloud) ---
def aggregate_models(client_weights_list):
    start_time = time.time(); global_weights = copy.deepcopy(client_weights_list[0]);
    for key in global_weights.keys():
        for i in range(1, len(client_weights_list)): global_weights[key] += client_weights_list[i][key]
        global_weights[key] = torch.div(global_weights[key], len(client_weights_list))
    return global_weights, time.time() - start_time

# --- HÀM MỚI: Logic cho Edge Server ---
def edge_server_update(edge_client_loaders, initial_model_state_dict, model_size_mb):
    """Mô phỏng toàn bộ quá trình tại một Edge Server trong MỘT global round."""
    
    # Khởi tạo mô hình cho Edge Server
    input_size = 64 if data_params['dataset_name'] == 'tiny_imagenet' else 32
    edge_model = SimpleCNN(num_classes=len(class_names), input_size=input_size).to(config.DEVICE)
    edge_model.load_state_dict(initial_model_state_dict)
    
    total_comp_cost = 0.0
    total_comm_cost = 0.0
    peak_ram = 0.0
    peak_gpu = 0.0
    
    # Bắt đầu vòng lặp Edge-Client
    for e_round in range(config.EDGE_ROUNDS):
        client_weights_list = []
        comp_cost_this_edge_round = 0.0
        
        # Gửi mô hình đến các client và nhận lại
        for client_loader in edge_client_loaders:
            # Tính 2x comm (downlink + uplink) cho mỗi client
            total_comm_cost += 2 * model_size_mb 
            
            w, c, r = client_update(client_loader, edge_model)
            client_weights_list.append(w)
            comp_cost_this_edge_round += c
            peak_ram = max(peak_ram, r['ram_peak_mb'])
            peak_gpu = max(peak_gpu, r['gpu_peak_mb'])
            
        # Tổng hợp tại Edge
        agg_w, agg_c = aggregate_models(client_weights_list)
        edge_model.load_state_dict(agg_w)
        
        total_comp_cost += (comp_cost_this_edge_round + agg_c)
    
    # Trả về mô hình cuối cùng của Edge và tổng chi phí
    resources = {"ram_peak_mb": peak_ram, "gpu_peak_mb": peak_gpu}
    return edge_model.state_dict(), total_comp_cost, total_comm_cost, resources

# --- Các hàm tiện ích (Không thay đổi) ---
def calculate_model_size_mb(model): return sum(p.nelement() * p.element_size() for p in model.parameters()) / (1024**2)
def evaluate_model(model, test_loader):
    model.eval(); test_loss, correct = 0, 0; criterion = nn.CrossEntropyLoss(reduction='sum');
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(config.DEVICE), target.to(config.DEVICE)
            output = model(data); test_loss += criterion(output, target).item(); correct += output.argmax(dim=1).eq(target).sum().item()
    return test_loss / len(test_loader.dataset), 100. * correct / len(test_loader.dataset)

In [11]:
# Cell 5: Vòng lặp huấn luyện chính (Cập nhật) 🚀
input_size = 64 if data_params['dataset_name'] == 'tiny_imagenet' else 32
global_model = SimpleCNN(num_classes=len(class_names), input_size=input_size).to(config.DEVICE)
model_size_mb = calculate_model_size_mb(global_model)
print(f"Kích thước mô hình: {model_size_mb:.4f} MB")

history = {'rounds': [], 'accuracy': [], 'loss': [], 'comm_cost': [], 'comp_cost': [], 'ram_usage': [], 'gpu_usage': []}
print(f"\n--- BẮT ĐẦU MÔ PHỎNG 3-TẦNG ({config.ALGORITHM.upper()}) ---")

# Vòng lặp GLOBAL (Cloud <-> Edge)
for g_round in range(config.GLOBAL_ROUNDS):
    round_start_time = time.time()
    
    edge_weights_list = []
    round_comp_cost = 0.0
    round_comm_cost = 0.0
    round_peak_ram = 0.0
    round_peak_gpu = 0.0
    
    global_model_state = global_model.state_dict()
    
    # --- Giai đoạn Edge Server ---
    for edge_id in range(config.NUM_EDGE):
        # 1. Giao tiếp Cloud -> Edge (Downlink)
        round_comm_cost += model_size_mb
        
        # Lấy nhóm client cho edge này
        edge_client_loaders = client_groups_for_edges[edge_id]
        
        # Chạy toàn bộ quá trình Edge (bao gồm EDGE_ROUNDS)
        final_edge_w, edge_comp, edge_comm, edge_res = edge_server_update(
            edge_client_loaders, global_model_state, model_size_mb
        )
        
        # 2. Giao tiếp Edge -> Cloud (Uplink)
        edge_weights_list.append(final_edge_w)
        round_comm_cost += model_size_mb
        
        # Tích lũy chi phí
        round_comp_cost += edge_comp
        round_comm_cost += edge_comm
        round_peak_ram = max(round_peak_ram, edge_res['ram_peak_mb'])
        round_peak_gpu = max(round_peak_gpu, edge_res['gpu_peak_mb'])

    # --- Giai đoạn Cloud Server ---
    # Tổng hợp các mô hình từ Edge
    global_weights, global_agg_cost = aggregate_models(edge_weights_list)
    global_model.load_state_dict(global_weights)
    round_comp_cost += global_agg_cost
    
    # --- Đánh giá & Ghi nhận ---
    test_loss, test_accuracy = evaluate_model(global_model, test_loader)
    round_elapsed_time = time.time() - round_start_time
    
    history['rounds'].append(g_round + 1); history['accuracy'].append(test_accuracy)
    history['loss'].append(test_loss); history['comm_cost'].append(round_comm_cost)
    history['comp_cost'].append(round_comp_cost); history['ram_usage'].append(round_peak_ram)
    history['gpu_usage'].append(round_peak_gpu)
    
    print(f"--- Global Vòng {g_round + 1:02d}/{config.GLOBAL_ROUNDS} ---")
    print(f"  Performance | Accuracy: {test_accuracy:.2f}% | Loss: {test_loss:.4f}")
    print(f"  Costs       | Computation: {round_comp_cost:.2f}s | Communication: {round_comm_cost:.2f}MB")
    print(f"  Resources   | Peak RAM (Client): {round_peak_ram:.2f}MB | Peak GPU Mem (Client): {round_peak_gpu:.2f}MB")
    print(f"  Round Time  | Total elapsed: {round_elapsed_time:.2f}s")

print("\n--- MÔ PHỎNG HOÀN TẤT ---")
shutdown_pynvml()

Kích thước mô hình: 8.0955 MB

--- BẮT ĐẦU MÔ PHỎNG 3-TẦNG (FEDPROX) ---
--- Global Vòng 01/10 ---
  Performance | Accuracy: 40.94% | Loss: 2.0209
  Costs       | Computation: 184.65s | Communication: 728.59MB
  Resources   | Peak RAM (Client): 601.97MB | Peak GPU Mem (Client): 0.00MB
  Round Time  | Total elapsed: 185.81s
--- Global Vòng 02/10 ---
  Performance | Accuracy: 52.83% | Loss: 1.4372
  Costs       | Computation: 181.52s | Communication: 728.59MB
  Resources   | Peak RAM (Client): 0.00MB | Peak GPU Mem (Client): 0.00MB
  Round Time  | Total elapsed: 182.70s
--- Global Vòng 03/10 ---
  Performance | Accuracy: 56.41% | Loss: 1.2741
  Costs       | Computation: 181.80s | Communication: 728.59MB
  Resources   | Peak RAM (Client): 0.00MB | Peak GPU Mem (Client): 0.00MB
  Round Time  | Total elapsed: 183.00s
--- Global Vòng 04/10 ---
  Performance | Accuracy: 57.21% | Loss: 1.2844
  Costs       | Computation: 176.10s | Communication: 728.59MB
  Resources   | Peak RAM (Client): 0.0

KeyboardInterrupt: 

In [ ]:
# Cell 6: Trực quan hóa kết quả 📊
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
title = (f"3-Tier FL: {config.ALGORITHM.upper()} | Dataset: {data_params['dataset_name']} ({data_params['distribution_mode']})\n"
         f"{config.NUM_EDGE} Edges | {config.GLOBAL_ROUNDS} G-Rounds | {config.EDGE_ROUNDS} E-Rounds")
fig.suptitle(title, fontsize=18)
rounds = history['rounds']
axes[0, 0].plot(rounds, history['accuracy'], marker='o'); axes[0, 0].set_title("Độ chính xác"); axes[0, 0].grid(True); axes[0, 0].set_ylim(bottom=0)
axes[0, 1].plot(rounds, history['loss'], marker='x', color='r'); axes[0, 1].set_title("Hàm mất mát"); axes[0, 1].grid(True)
axes[1, 0].plot(rounds, np.cumsum(history['comp_cost']), marker='s', color='g'); axes[1, 0].set_title("Chi phí tính toán tích lũy"); axes[1, 0].grid(True)
axes[1, 1].plot(rounds, np.cumsum(history['comm_cost']), marker='d', color='purple'); axes[1, 1].set_title("Chi phí giao tiếp tích lũy"); axes[1, 1].grid(True)
for ax in axes.flat: ax.set_xlabel("Vòng Global")
plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()